# ToxiLens API Usage Examples

This notebook demonstrates how to use the ToxiLens API for various toxicity prediction tasks.

**Prerequisites:**
- ToxiLens backend running at `http://localhost:8000`
- Python packages: `requests`, `pandas`, `matplotlib`

**Setup:**
```bash
pip install requests pandas matplotlib
cd backend
uvicorn app.main:app --reload --port 8000
```

In [ ]:
import requests
import pandas as pd
import json
from typing import Dict, List
import matplotlib.pyplot as plt

# API base URL
BASE_URL = "http://localhost:8000"

# Helper function to pretty print JSON
def print_json(data: dict):
    print(json.dumps(data, indent=2))

## 1. Health Check

Verify the API is running and models are loaded.

In [ ]:
response = requests.get(f"{BASE_URL}/health")
print(f"Status Code: {response.status_code}")
print_json(response.json())

## 2. Single Molecule Prediction

Predict toxicity for a single molecule (Aspirin).

In [ ]:
# Aspirin SMILES
smiles = "CC(=O)Oc1ccccc1C(=O)O"

# Make prediction request
response = requests.post(
    f"{BASE_URL}/predict",
    json={
        "smiles": smiles,
        "include_shap": True,
        "include_heatmap": True,
        "include_admet": True
    }
)

data = response.json()

# Print key results
print(f"Compound: {data.get('compound_name', 'Unknown')}")
print(f"SMILES: {data['smiles']}")
print(f"Risk Level: {data['risk_level']}")
print(f"Composite Risk Score: {data['composite_risk']:.3f}")
print("\nTox21 Predictions:")
for assay, pred in data['predictions'].items():
    print(f"  {assay}: {pred['probability']:.3f} ({pred['uncertainty']})")

### View SHAP Feature Importance

In [ ]:
print("\nTop 10 SHAP Features:")
for i, feature in enumerate(data['shap_top10'][:10], 1):
    direction = "🔴 TOXIC" if feature['direction'] == 'toxic' else "🔵 PROTECT"
    print(f"{i}. {feature['feature']:<20} = {feature['value']:>8.2f}  SHAP: {feature['shap']:>+7.3f}  {direction}")

### View Structural Alerts

In [ ]:
print("\nStructural Alerts:")
if data['alerts']:
    for alert in data['alerts']:
        severity_emoji = {"HIGH": "🔴", "MEDIUM": "🟠", "LOW": "🟡"}.get(alert['severity'], "⚪")
        print(f"{severity_emoji} {alert['severity']:<6} {alert['name']:<30} {alert['description']}")
else:
    print("  ✅ No structural alerts detected")

### View ADMET Properties

In [ ]:
print("\nADMET Properties:")
admet = data['admet']
print(f"  QED (Drug-likeness):        {admet['qed']:.3f}")
print(f"  Lipinski Violations:        {admet['lipinski_violations']}")
print(f"  Molecular Weight:           {admet['mw']:.1f} Da")
print(f"  logP:                       {admet['logp']:.2f}")
print(f"  TPSA:                       {admet['tpsa']:.1f} Ų")
print(f"  BBB Penetration:            {admet['bbb_penetration']}")
print(f"  Oral Bioavailability:       {admet['oral_bioavailability']}")
print(f"  hERG Inhibition Risk:       {admet['herg_inhibition']:.3f}")

## 3. Batch Virtual Screening

Screen multiple molecules from a CSV file.

In [ ]:
# Upload CSV file for batch screening
with open('../examples/batch_screening_example.csv', 'rb') as f:
    files = {'file': ('batch_screening_example.csv', f, 'text/csv')}
    response = requests.post(
        f"{BASE_URL}/predict_batch",
        files=files,
        data={
            'risk_threshold': 0.5,
            'sort_by': 'composite_risk'
        }
    )

batch_results = response.json()

# Convert to DataFrame for analysis
df = pd.DataFrame(batch_results['results'])

print(f"Processed {len(df)} compounds")
print(f"\nRisk Distribution:")
print(df['risk_level'].value_counts())

# Display top 10 highest risk compounds
print("\nTop 10 Highest Risk Compounds:")
print(df[['compound_id', 'compound_name', 'composite_risk', 'risk_level']].head(10))

### Visualize Risk Distribution

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df['composite_risk'], bins=20, edgecolor='black', alpha=0.7)
plt.axvline(0.35, color='green', linestyle='--', label='Low/Medium threshold')
plt.axvline(0.6, color='red', linestyle='--', label='Medium/High threshold')
plt.xlabel('Composite Risk Score')
plt.ylabel('Number of Compounds')
plt.title('Batch Screening: Risk Score Distribution')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 4. Chemical Space Exploration

Find similar molecules in the Tox21 dataset.

In [ ]:
# Find molecules similar to Aspirin
smiles = "CC(=O)Oc1ccccc1C(=O)O"

response = requests.get(
    f"{BASE_URL}/similar",
    params={'smiles': smiles, 'top_k': 10}
)

similar = response.json()

print("Top 10 Similar Molecules:")
for i, mol in enumerate(similar['similar_molecules'], 1):
    print(f"{i}. Tanimoto: {mol['tanimoto_similarity']:.3f}")
    print(f"   SMILES: {mol['smiles']}")
    print(f"   Toxicity: {mol['toxicity_labels']}")
    print()

## 5. What-If Analysis

Compare toxicity before and after molecular modification.

In [ ]:
# Original: Aspirin (ester)
original_smiles = "CC(=O)Oc1ccccc1C(=O)O"

# Modified: Replace ester with amide
modified_smiles = "CC(=O)Nc1ccccc1C(=O)O"

response = requests.post(
    f"{BASE_URL}/what_if",
    json={
        'original_smiles': original_smiles,
        'modified_smiles': modified_smiles
    }
)

comparison = response.json()

print("What-If Analysis: Ester → Amide Modification")
print(f"\nOriginal Risk: {comparison['original']['composite_risk']:.3f}")
print(f"Modified Risk: {comparison['modified']['composite_risk']:.3f}")
print(f"Delta Risk: {comparison['delta_composite_risk']:+.3f}")

print("\nPer-Assay Changes:")
for assay, delta in comparison['delta_predictions'].items():
    direction = "⬆️ WORSE" if delta > 0.1 else "⬇️ BETTER" if delta < -0.1 else "➡️ SIMILAR"
    print(f"  {assay:<15} {delta:+.3f}  {direction}")

## 6. De-Risking Lab

Generate bioisostere variants to reduce toxicity.

In [ ]:
# Toxic molecule: Bisphenol A
smiles = "CC(C)(c1ccc(O)cc1)c1ccc(O)cc1"

response = requests.post(
    f"{BASE_URL}/derisk",
    json={
        'smiles': smiles,
        'target_assays': ['NR-ER', 'NR-AR'],
        'n_variants': 8,
        'strategy': ['bioisostere', 'removal']
    }
)

derisk = response.json()

print(f"Original Compound Risk: {derisk['original_risk']:.3f}")
print(f"\nGenerated {len(derisk['variants'])} De-Risked Variants:\n")

for i, variant in enumerate(derisk['variants'], 1):
    print(f"{i}. Modification: {variant['modification']}")
    print(f"   SMILES: {variant['smiles']}")
    print(f"   New Risk: {variant['composite_risk']:.3f}")
    print(f"   Delta Risk: {variant['delta_risk']:+.3f}")
    if 'rationale' in variant:
        print(f"   Rationale: {variant['rationale']}")
    print()

## 7. Generate LLM Assessment Report

Create a comprehensive toxicity report with AI-generated insights.

In [ ]:
# First get prediction data
smiles = "COc1cccc2c1C(=O)c1c(O)c3c(c(O)c1C2=O)CC(O)(C(=O)CO)CC3OC1CC(N)C(O)C(C)O1"

pred_response = requests.post(
    f"{BASE_URL}/predict",
    json={'smiles': smiles, 'include_shap': True, 'include_admet': True}
)

prediction_data = pred_response.json()

# Generate report
report_response = requests.post(
    f"{BASE_URL}/generate_report",
    json={
        'smiles': smiles,
        'prediction_data': prediction_data,
        'format': 'text',
        'audience': 'medicinal_chemist'
    }
)

report = report_response.json()

print("=" * 80)
print("TOXILENS ASSESSMENT REPORT")
print("=" * 80)
print(report['report_text'])
print("=" * 80)

### Download PDF Report

In [ ]:
# Generate PDF report
pdf_response = requests.post(
    f"{BASE_URL}/generate_report",
    json={
        'smiles': smiles,
        'prediction_data': prediction_data,
        'format': 'pdf',
        'audience': 'medicinal_chemist'
    }
)

# Save PDF to file
with open('toxicity_report.pdf', 'wb') as f:
    f.write(pdf_response.content)

print("PDF report saved to: toxicity_report.pdf")

## 8. Multi-Molecule Comparison

Compare toxicity profiles of multiple molecules side-by-side.

In [ ]:
# Compare 3 NSAIDs
molecules = [
    {"name": "Aspirin", "smiles": "CC(=O)Oc1ccccc1C(=O)O"},
    {"name": "Ibuprofen", "smiles": "CC(C)Cc1ccc(cc1)C(C)C(=O)O"},
    {"name": "Naproxen", "smiles": "COc1ccc2cc(ccc2c1)C(C)C(=O)O"}
]

# Get predictions for all molecules
predictions = []
for mol in molecules:
    response = requests.post(
        f"{BASE_URL}/predict",
        json={'smiles': mol['smiles']}
    )
    pred = response.json()
    pred['name'] = mol['name']
    predictions.append(pred)

# Create comparison table
assays = list(predictions[0]['predictions'].keys())

print("Multi-Molecule Comparison: NSAIDs\n")
print(f"{'Assay':<15}", end="")
for pred in predictions:
    print(f"{pred['name']:<15}", end="")
print()
print("-" * (15 + 15 * len(predictions)))

for assay in assays:
    print(f"{assay:<15}", end="")
    for pred in predictions:
        prob = pred['predictions'][assay]['probability']
        print(f"{prob:<15.3f}", end="")
    print()

print("\nComposite Risk:")
for pred in predictions:
    print(f"  {pred['name']:<15} {pred['composite_risk']:.3f} ({pred['risk_level']})")

## 9. Error Handling Examples

Demonstrate proper error handling for common issues.

In [ ]:
# Example 1: Invalid SMILES
try:
    response = requests.post(
        f"{BASE_URL}/predict",
        json={'smiles': 'INVALID_SMILES_123'}
    )
    if response.status_code != 200:
        error = response.json()
        print(f"Error {response.status_code}: {error['detail']}")
except Exception as e:
    print(f"Request failed: {e}")

# Example 2: Empty SMILES
try:
    response = requests.post(
        f"{BASE_URL}/predict",
        json={'smiles': ''}
    )
    if response.status_code != 200:
        error = response.json()
        print(f"Error {response.status_code}: {error['detail']}")
except Exception as e:
    print(f"Request failed: {e}")

# Example 3: Missing required field
try:
    response = requests.post(
        f"{BASE_URL}/predict",
        json={}
    )
    if response.status_code != 200:
        error = response.json()
        print(f"Error {response.status_code}: {error['detail']}")
except Exception as e:
    print(f"Request failed: {e}")

## Summary

This notebook demonstrated:

1. ✅ Health check and API connectivity
2. ✅ Single molecule prediction with SHAP, alerts, and ADMET
3. ✅ Batch virtual screening from CSV
4. ✅ Chemical space exploration and similarity search
5. ✅ What-if analysis for molecular modifications
6. ✅ De-risking lab with bioisostere generation
7. ✅ LLM-powered assessment reports (text and PDF)
8. ✅ Multi-molecule comparison
9. ✅ Error handling best practices

For more information:
- **API Documentation:** http://localhost:8000/docs
- **GitHub:** https://github.com/your-handle/toxilens
- **Model Card:** docs/model_card.md